# DenseNet121 — Kaş ROI Deepfake Sınıflandırma (Stabilized V3)

Bu notebook, **önceden kırpılmış kaş ROI framelerini** kullanarak DenseNet121 ile Real/Fake deepfake sınıflandırması yapar.

### Okunan veri
`MyDrive / AISC DeepFake Çalışmaları / Deneyler / Nazlıcan / Deney 1 / Kaş`

Notebook **Kaş klasöründeki crop görsellerini değiştirmez, yeniden kırpmaz, taşımaz veya yeniden adlandırmaz**.

Beklenen fiziksel veri düzeni:

```text
Kaş/
├── Real/
│   ├── train/
│   ├── val/
│   └── test/
└── Fake/
    ├── train/
    ├── val/
    └── test/
```

### Split politikası
Fiziksel train/val/test klasörleri audit için okunur. Model eğitimi için ROI'ler, kaynak video kimliği kullanılarak
**source-group seviyesinde seed=42 ile %80 Train / %10 Val / %10 Test** olarak bellekte yeniden atanır.

- Real ve Fake sınıfları bağımsız bölünür.
- Frame düzeyinde rastgele split yapılmaz.
- Aynı FF++ kaynak video çifti farklı manipülasyon yöntemlerinde görünse bile aynı split grubunda tutulur.
- Aynı piksel içeriğine sahip tekrar görseller varsa, aynı sınıf içinde otomatik olarak aynı split bileşenine birleştirilir.
- Aynı görüntü hash'i Real ve Fake etiketlerinde birden bulunursa veri etiketi çelişkisi olarak eğitim durdurulur.

### Çıktı
`Nazlıcan / Deney 1 / Sonuçlar / DenseNet121_Kas_Results_<RUN_ID>/`

```text
DenseNet121_Kas_Results_<RUN_ID>/
├── checkpoints/
├── logs/
├── metrics/
├── predictions/
├── figures/
├── artifacts/
├── config_resolved.yaml
├── environment.json
├── requirements_lock.txt
├── run_summary.json
└── output_manifest.csv
```

### V3 kararlılık düzeltmeleri
- Keras `train_on_batch()` sonucu `return_dict=True` ile okunur; liste/skaler sürüm farkı sorun çıkarmaz.
- Binary label şekli model çıktısıyla uyumlu olacak şekilde `(batch, 1)` tutulur.
- Augmentation sonrası piksel aralığı 0–255'e clip edilir.
- `tf.data` map sırası deterministik tutulur.
- `.keras` checkpoint kaydı güncel Keras formatına göre yapılır.
- Resume sırasında CSV logları append edilir ve epoch sayısı logdan geri kazanılır.
- Final audit ve output manifest sıralaması düzeltilmiştir.


In [1]:
# ============================================================
# CELL 1 — ENVIRONMENT, IMPORTS, REPRODUCIBILITY
# ============================================================

import os
import sys
import gc
import json
import math
import random
import re
import hashlib
import platform
import subprocess
import unicodedata
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import yaml

from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    matthews_corrcoef,
)

warnings.filterwarnings("default")

SEED = 42

# Set reproducibility controls before model construction.
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
    DETERMINISM_ENABLED = True
except Exception as exc:
    DETERMINISM_ENABLED = False
    print("Deterministic ops could not be fully enabled:", exc)

GPUS = tf.config.list_physical_devices("GPU")

print("=" * 88)
print("DENSENET121 EYEBROW ROI DEEPFAKE EXPERIMENT — STABILIZED V3")
print("=" * 88)
print("Python              :", sys.version.split()[0])
print("TensorFlow          :", tf.__version__)
print("Keras               :", getattr(tf.keras, "__version__", "bundled-with-tensorflow"))
print("GPU count           :", len(GPUS))
print("Deterministic ops   :", DETERMINISM_ENABLED)
print("Seed                :", SEED)

if not GPUS:
    print("WARNING: GPU was not detected. The notebook can run on CPU, but training will be slow.")


DENSENET121 EYEBROW ROI DEEPFAKE EXPERIMENT — STABILIZED V3
Python              : 3.12.13
TensorFlow          : 2.20.0
Keras               : 3.13.2
GPU count           : 1
Deterministic ops   : True
Seed                : 42


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [2]:
# ============================================================
# CELL 2 — GOOGLE DRIVE AND EXACT PROJECT PATHS
# ============================================================

from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")


def normalize_name(value):
    return unicodedata.normalize("NFC", str(value)).strip().casefold()


def find_child_folder(parent, *expected_names):
    parent = Path(parent)
    if not parent.exists():
        raise FileNotFoundError(f"Parent folder not found: {parent}")

    expected = {normalize_name(name) for name in expected_names}

    for item in parent.iterdir():
        if item.is_dir() and normalize_name(item.name) in expected:
            return item

    available = sorted(item.name for item in parent.iterdir() if item.is_dir())
    raise FileNotFoundError(
        "Required folder was not found.\n"
        f"Parent: {parent}\n"
        f"Expected one of: {list(expected_names)}\n"
        f"Available folders: {available}"
    )


MY_DRIVE = Path("/content/drive/MyDrive")

AISC_ROOT = find_child_folder(
    MY_DRIVE,
    "AISC DeepFake Çalışmaları",
    "AISC Deepfake Çalışmaları",
)

EXPERIMENTS_ROOT = find_child_folder(AISC_ROOT, "Deneyler")
NAZLICAN_ROOT = find_child_folder(EXPERIMENTS_ROOT, "Nazlıcan", "Nazlican")
EXPERIMENT_ROOT = find_child_folder(NAZLICAN_ROOT, "Deney 1", "Deney1")
EYEBROW_ROOT = find_child_folder(EXPERIMENT_ROOT, "Kaş", "kaş", "Kas", "kas")

# Source-frame selection metadata lives here and is used only for audit/leakage control.
FRAME_ROOT = find_child_folder(EXPERIMENTS_ROOT, "Deney 1 Frame", "Deney1 Frame")

# Only this output folder may be created/modified by the training notebook.
try:
    RESULTS_ROOT = find_child_folder(EXPERIMENT_ROOT, "Sonuçlar", "sonuçlar", "Sonuclar")
except FileNotFoundError:
    RESULTS_ROOT = EXPERIMENT_ROOT / "Sonuçlar"
    RESULTS_ROOT.mkdir(parents=False, exist_ok=False)

assert EYEBROW_ROOT.exists()
assert RESULTS_ROOT.exists()
assert EYEBROW_ROOT.resolve() != RESULTS_ROOT.resolve()

print("AISC root       :", AISC_ROOT)
print("Experiments root:", EXPERIMENTS_ROOT)
print("Nazlıcan root   :", NAZLICAN_ROOT)
print("Experiment root :", EXPERIMENT_ROOT)
print("Eyebrow input   :", EYEBROW_ROOT)
print("Frame metadata  :", FRAME_ROOT)
print("Results root    :", RESULTS_ROOT)

Mounted at /content/drive
AISC root       : /content/drive/MyDrive/AISC DeepFake Çalışmaları
Experiments root: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler
Nazlıcan root   : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan
Experiment root : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1
Eyebrow input   : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş
Frame metadata  : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Deney 1 Frame
Results root    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar


In [3]:
# ============================================================
# CELL 3 — EXPERIMENT CONFIGURATION AND STANDARD OUTPUT TREE
# ============================================================

CONFIG = {
    "seed": SEED,
    "region": "eyebrow",
    "model": "DenseNet121",
    "framework": "TensorFlow/Keras",
    "image_size": [224, 224],
    "batch_size": 16,
    "frozen_epochs": 12,
    "finetune_epochs": 10,
    "dropout_rate": 0.40,
    "frozen_learning_rate": 1e-3,
    "finetune_learning_rate": 1e-5,
    "early_stopping_patience": 4,
    "reduce_lr_patience": 2,
    "label_mapping": {"real": 0, "fake": 1},
    "model_selection_metric": "val_auc",
    "model_selection_mode": "max",
    "pretrained_weights": "imagenet",
    "resize_policy": "resize_with_pad_preserve_aspect_ratio",
    "split_policy": "source_group_80_10_10_real_fake_independent",
}

# Historical ROI extraction counts. These are audit references only.
EXPECTED_ROI_COUNTS = {
    ("real", "train"): 786,
    ("real", "val"): 106,
    ("real", "test"): 101,
    ("fake", "train"): 776,
    ("fake", "val"): 98,
    ("fake", "test"): 95,
}

BASE_RUN_ID = (
    datetime.now().strftime("%Y%m%d_%H%M")
    + "_eyebrow_densenet121_seed42"
)
RUN_ID = BASE_RUN_ID
RUN_ROOT = RESULTS_ROOT / f"DenseNet121_Kas_Results_{RUN_ID}"

# Never overwrite a previous run, including a previously failed run.
run_counter = 2
while RUN_ROOT.exists():
    RUN_ID = f"{BASE_RUN_ID}_r{run_counter:02d}"
    RUN_ROOT = RESULTS_ROOT / f"DenseNet121_Kas_Results_{RUN_ID}"
    run_counter += 1

CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
LOG_DIR = RUN_ROOT / "logs"
METRICS_DIR = RUN_ROOT / "metrics"
PREDICTION_DIR = RUN_ROOT / "predictions"
FIGURE_DIR = RUN_ROOT / "figures"
ARTIFACT_DIR = RUN_ROOT / "artifacts"

for directory in [
    CHECKPOINT_DIR,
    LOG_DIR,
    METRICS_DIR,
    PREDICTION_DIR,
    FIGURE_DIR,
    ARTIFACT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=False)

CONFIG["run_id"] = RUN_ID
CONFIG["input_root"] = str(EYEBROW_ROOT)
CONFIG["results_root"] = str(RESULTS_ROOT)
CONFIG["run_root"] = str(RUN_ROOT)

with open(RUN_ROOT / "config_resolved.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, sort_keys=False, allow_unicode=True)

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "tensorflow": tf.__version__,
    "keras": str(getattr(tf.keras, "__version__", "bundled-with-tensorflow")),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "gpu_devices": [str(gpu) for gpu in GPUS],
    "seed": SEED,
    "deterministic_ops": DETERMINISM_ENABLED,
}
with open(RUN_ROOT / "environment.json", "w", encoding="utf-8") as f:
    json.dump(environment, f, indent=2, ensure_ascii=False)

requirements_text = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True,
)
(RUN_ROOT / "requirements_lock.txt").write_text(
    requirements_text,
    encoding="utf-8",
)

print("RUN ID  :", RUN_ID)
print("RUN ROOT:", RUN_ROOT)


RUN ID  : 20260808_1335_eyebrow_densenet121_seed42
RUN ROOT: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/DenseNet121_Kas_Results_20260808_1335_eyebrow_densenet121_seed42


## Veri okuma politikası

Kaş ROI çıkarma aşamasında oluşan görüntüler **ham model girdisi** kabul edilir. Notebook yalnızca bellekte:
1. en-boy oranını koruyarak `224×224` letterbox/resize uygular,
2. train kümesine sınırlı augmentation uygular,
3. DenseNet121 `preprocess_input` dönüşümünü uygular.

Kaynak `Kaş` klasörüne hiçbir şey yazılmaz.

In [4]:
# ============================================================
# CELL 4 — DISCOVER EYEBROW IMAGES AND AUDIT COUNTS
# ============================================================

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def collect_images(folder):
    return sorted(
        path
        for path in Path(folder).rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


CLASS_DIRS = {
    "real": find_child_folder(EYEBROW_ROOT, "Real", "real"),
    "fake": find_child_folder(EYEBROW_ROOT, "Fake", "fake"),
}

records = []

for label_name, class_dir in CLASS_DIRS.items():
    for split_name in ["train", "val", "test"]:
        split_dir = find_child_folder(class_dir, split_name)
        image_files = collect_images(split_dir)

        if not image_files:
            raise RuntimeError(
                f"No images found in required folder: {split_dir}"
            )

        for image_path in image_files:
            records.append(
                {
                    "path": str(image_path),
                    "file_name": image_path.name,
                    "label": label_name,
                    "label_id": CONFIG["label_mapping"][label_name],
                    "split": split_name,
                }
            )

images_df = pd.DataFrame(records)

assert not images_df.empty
assert set(images_df["label"]) == {"real", "fake"}
assert set(images_df["split"]) == {"train", "val", "test"}

# Duplicate filename check matters because metadata joins use stable filenames.
duplicate_names = images_df["file_name"].duplicated(keep=False)
if duplicate_names.any():
    duplicates = images_df.loc[duplicate_names, "file_name"].unique()[:20]
    raise RuntimeError(
        "Duplicate ROI filenames were found. Metadata matching would be ambiguous.\n"
        f"Examples: {duplicates.tolist()}"
    )

count_table = (
    images_df.groupby(["label", "split"])
    .size()
    .rename("count")
    .reset_index()
)

print(count_table.to_string(index=False))
print("\nTotal usable eyebrow ROI images:", len(images_df))

count_audit = []
for row in count_table.itertuples(index=False):
    expected = EXPECTED_ROI_COUNTS.get((row.label, row.split))
    matches_reference = (expected == row.count) if expected is not None else None
    count_audit.append(
        {
            "label": row.label,
            "split": row.split,
            "actual_count": int(row.count),
            "reference_count": expected,
            "matches_roi_report": matches_reference,
        }
    )

count_audit_df = pd.DataFrame(count_audit)
count_audit_df.to_csv(
    ARTIFACT_DIR / "dataset_count_audit.csv",
    index=False,
)

if not count_audit_df["matches_roi_report"].all():
    print(
        "\nWARNING: Current image counts differ from the historical ROI report. "
        "Training may still continue if all quality gates below pass."
    )
else:
    print("\nROI count audit: PASS")

label split  count
 fake  test     95
 fake train    776
 fake   val     98
 real  test    101
 real train    786
 real   val    106

Total usable eyebrow ROI images: 1962

ROI count audit: PASS


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
# ============================================================
# CELL 5 — SOURCE MAPPING + HASH-AWARE SOURCE-GROUP 80/10/10 RESPLIT
# ============================================================
# The physical Kaş/train-val-test folders are an audit trail.
# No input image is moved or modified. Training split assignment is
# rebuilt in memory at source-group level.

SELECTION_METADATA_PATH = FRAME_ROOT / "secim_metadata.csv"

if not SELECTION_METADATA_PATH.exists():
    raise FileNotFoundError(
        "Required source mapping file was not found:\n"
        f"{SELECTION_METADATA_PATH}\n"
        "Training is stopped because source-video identity must not be guessed."
    )

selection_df = pd.read_csv(SELECTION_METADATA_PATH)

required_selection_columns = {
    "sinif",
    "split",
    "orijinal_yol",
    "yeni_yol",
    "dosya_adi",
}
missing_columns = required_selection_columns - set(selection_df.columns)

if missing_columns:
    raise RuntimeError(
        "secim_metadata.csv schema is not compatible.\n"
        f"Missing columns: {sorted(missing_columns)}\n"
        f"Available columns: {selection_df.columns.tolist()}"
    )

selection_df = selection_df.copy()
selection_df["frame_file_name"] = selection_df["dosya_adi"].astype(str)
selection_df["source_video"] = selection_df["orijinal_yol"].map(
    lambda value: Path(str(value)).parent.name
)
selection_df["original_frame_name"] = selection_df["orijinal_yol"].map(
    lambda value: Path(str(value)).name
)
selection_df["source_label"] = selection_df["sinif"].astype(str).str.casefold()
selection_df["source_split"] = selection_df["split"].astype(str).str.casefold()

frame_index_extracted = (
    selection_df["original_frame_name"]
    .str.extract(r"(\d+)(?=\.[^.]+$)", expand=False)
)
selection_df["frame_index"] = pd.to_numeric(
    frame_index_extracted,
    errors="coerce",
).astype("Int64")

if selection_df["frame_file_name"].duplicated().any():
    duplicate_examples = (
        selection_df.loc[
            selection_df["frame_file_name"].duplicated(keep=False),
            "frame_file_name",
        ]
        .head(20)
        .tolist()
    )
    raise RuntimeError(
        "secim_metadata.csv contains duplicate renamed frame filenames.\n"
        f"Examples: {duplicate_examples}"
    )

if selection_df["source_video"].isna().any():
    raise RuntimeError("secim_metadata.csv contains missing source_video values.")

# ------------------------------------------------------------
# Direct mapping: ROI output kept the selected-frame basename.
# ------------------------------------------------------------
manifest_df = images_df.merge(
    selection_df[
        [
            "frame_file_name",
            "source_video",
            "frame_index",
            "source_label",
            "source_split",
            "orijinal_yol",
        ]
    ],
    left_on="file_name",
    right_on="frame_file_name",
    how="left",
    validate="one_to_one",
)

direct_match_rate = float(manifest_df["source_video"].notna().mean())
print(f"Direct ROI-to-frame metadata match rate: {direct_match_rate:.2%}")

# ------------------------------------------------------------
# Fallback: Kaş/metadata.csv maps ROI filename -> source frame filename.
# ------------------------------------------------------------
if direct_match_rate < 1.0:
    EYEBROW_METADATA_PATH = EYEBROW_ROOT / "metadata.csv"

    if not EYEBROW_METADATA_PATH.exists():
        missing_examples = manifest_df.loc[
            manifest_df["source_video"].isna(),
            "file_name",
        ].head(20).tolist()
        raise RuntimeError(
            "Not every ROI file could be mapped directly to secim_metadata.csv, "
            "and Kaş/metadata.csv was not found.\n"
            f"Unmatched ROI examples: {missing_examples}\n"
            "Training is stopped instead of inventing source_video."
        )

    eyebrow_meta = pd.read_csv(EYEBROW_METADATA_PATH)
    lower_columns = {str(col).casefold(): col for col in eyebrow_meta.columns}

    def detect_column(candidates):
        for candidate in candidates:
            if candidate.casefold() in lower_columns:
                return lower_columns[candidate.casefold()]
        return None

    output_col = detect_column(
        ["output_path", "roi_path", "eyebrow_path", "crop_path", "saved_path"]
    )
    source_col = detect_column(
        ["input_path", "source_path", "source_frame", "frame_path", "original_path"]
    )
    status_col = detect_column(["status", "state"])

    if output_col is None or source_col is None:
        raise RuntimeError(
            "Kaş/metadata.csv exists, but ROI/source path columns could not be detected.\n"
            f"Columns: {eyebrow_meta.columns.tolist()}"
        )

    if status_col is not None:
        eyebrow_meta = eyebrow_meta.loc[
            eyebrow_meta[status_col].astype(str).str.upper() == "SUCCESS"
        ].copy()

    eyebrow_meta["roi_file_name"] = eyebrow_meta[output_col].map(
        lambda value: Path(str(value)).name
    )
    eyebrow_meta["frame_file_name"] = eyebrow_meta[source_col].map(
        lambda value: Path(str(value)).name
    )

    if eyebrow_meta["roi_file_name"].duplicated().any():
        raise RuntimeError("Duplicate ROI filenames exist in Kaş/metadata.csv.")

    roi_to_frame = eyebrow_meta[
        ["roi_file_name", "frame_file_name"]
    ].drop_duplicates()

    manifest_df = (
        images_df.merge(
            roi_to_frame,
            left_on="file_name",
            right_on="roi_file_name",
            how="left",
            validate="one_to_one",
        )
        .merge(
            selection_df[
                [
                    "frame_file_name",
                    "source_video",
                    "frame_index",
                    "source_label",
                    "source_split",
                    "orijinal_yol",
                ]
            ],
            on="frame_file_name",
            how="left",
            validate="one_to_one",
        )
    )

if manifest_df["source_video"].isna().any():
    examples = manifest_df.loc[
        manifest_df["source_video"].isna(),
        "file_name",
    ].head(20).tolist()
    raise RuntimeError(
        "Some eyebrow ROI images still have no source_video mapping.\n"
        f"Examples: {examples}"
    )

manifest_df["source_label"] = manifest_df["source_label"].astype(str).str.casefold()
manifest_df["source_split"] = manifest_df["source_split"].astype(str).str.casefold()

label_mismatch = manifest_df["label"] != manifest_df["source_label"]
folder_metadata_split_mismatch = manifest_df["split"] != manifest_df["source_split"]

if label_mismatch.any():
    raise RuntimeError(
        "Class labels in Kaş folders disagree with secim_metadata.csv."
    )
if folder_metadata_split_mismatch.any():
    raise RuntimeError(
        "Physical train/val/test folders in Kaş disagree with secim_metadata.csv."
    )

manifest_df["original_split"] = manifest_df["split"].astype(str)

# ------------------------------------------------------------
# Hash each ROI before splitting.
# This lets identical pixel content be kept in the same split by construction.
# ------------------------------------------------------------
def sha256_file(file_path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(file_path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


print("Calculating image SHA-256 values...")
manifest_df["sha256"] = [
    sha256_file(path)
    for path in manifest_df["path"]
]

# The same exact image may not carry contradictory labels.
hash_label_counts = manifest_df.groupby("sha256")["label"].nunique()
conflicting_hashes = hash_label_counts[hash_label_counts > 1]
if not conflicting_hashes.empty:
    examples = conflicting_hashes.index[:20].tolist()
    raise RuntimeError(
        "DATA LABEL CONFLICT: identical image content occurs with both Real and Fake labels.\n"
        f"Conflicting hash count: {len(conflicting_hashes)}\n"
        f"Examples: {examples}"
    )

# ------------------------------------------------------------
# Canonical source group.
# ------------------------------------------------------------
FFPP_PATTERN = re.compile(
    r"^(?:Deepfakes|FaceSwap|Face2Face|NeuralTextures|FaceShifter)"
    r"_(\d{3})_(\d{3})$"
)
DFD_PATTERN = re.compile(
    r"^DeepFakeDetection_(\d{2})_(\d{2})__"
)


def derive_source_group(row):
    source_video = str(row["source_video"])
    label = str(row["label"]).casefold()

    if label == "real":
        return f"real_video::{source_video}"

    ffpp_match = FFPP_PATTERN.fullmatch(source_video)
    if ffpp_match:
        left_id, right_id = sorted(ffpp_match.groups())
        return f"fake_ffpp_pair::{left_id}_{right_id}"

    dfd_match = DFD_PATTERN.match(source_video)
    if dfd_match:
        left_id, right_id = sorted(dfd_match.groups())
        return f"fake_dfd_pair::{left_id}_{right_id}"

    return f"fake_video::{source_video}"


manifest_df["source_group"] = manifest_df.apply(
    derive_source_group,
    axis=1,
)

if manifest_df["source_group"].isna().any():
    raise RuntimeError("Source-group derivation produced missing values.")

# ------------------------------------------------------------
# Merge source groups that share identical content.
# Union-find prevents A--B and B--C duplicate chains from being split apart.
# ------------------------------------------------------------
all_source_groups = sorted(manifest_df["source_group"].astype(str).unique())
parent = {group: group for group in all_source_groups}


def uf_find(group):
    root = group
    while parent[root] != root:
        root = parent[root]
    while parent[group] != group:
        next_group = parent[group]
        parent[group] = root
        group = next_group
    return root


def uf_union(group_a, group_b):
    root_a = uf_find(group_a)
    root_b = uf_find(group_b)
    if root_a == root_b:
        return
    if root_a < root_b:
        parent[root_b] = root_a
    else:
        parent[root_a] = root_b


for _, duplicate_frame in manifest_df.groupby("sha256"):
    duplicate_groups = sorted(
        duplicate_frame["source_group"].astype(str).unique()
    )
    if len(duplicate_groups) > 1:
        anchor = duplicate_groups[0]
        for other_group in duplicate_groups[1:]:
            uf_union(anchor, other_group)

component_members = {}
for group in all_source_groups:
    root = uf_find(group)
    component_members.setdefault(root, []).append(group)

group_to_split_group = {}
for root, members in component_members.items():
    members = sorted(members)
    component_token = "|".join(members)
    component_hash = hashlib.sha256(
        component_token.encode("utf-8")
    ).hexdigest()[:16]
    for group in members:
        group_to_split_group[group] = f"component::{component_hash}"

manifest_df["split_group"] = manifest_df["source_group"].map(
    group_to_split_group
)

if manifest_df["split_group"].isna().any():
    raise RuntimeError("Hash-aware split-group construction failed.")

duplicate_cluster_rows = []
for split_group, frame in manifest_df.groupby("split_group"):
    source_group_count = frame["source_group"].nunique()
    if source_group_count > 1:
        duplicate_cluster_rows.append(
            {
                "split_group": split_group,
                "source_group_count": int(source_group_count),
                "image_count": int(len(frame)),
                "source_groups": " | ".join(
                    sorted(frame["source_group"].astype(str).unique())
                ),
            }
        )

pd.DataFrame(
    duplicate_cluster_rows,
    columns=[
        "split_group",
        "source_group_count",
        "image_count",
        "source_groups",
    ],
).to_csv(
    ARTIFACT_DIR / "content_duplicate_clusters.csv",
    index=False,
)

# ------------------------------------------------------------
# Real/Fake are split independently at split_group level.
# ------------------------------------------------------------
new_split = pd.Series(index=manifest_df.index, dtype="object")


def assign_grouped_split(class_frame, class_name):
    class_frame = class_frame.copy()
    unique_groups = int(class_frame["split_group"].nunique())

    if unique_groups < 10:
        raise RuntimeError(
            f"Too few leak-proof source groups for 80/10/10 split in {class_name}: "
            f"{unique_groups}"
        )

    first_splitter = GroupShuffleSplit(
        n_splits=1,
        train_size=0.80,
        random_state=SEED,
    )
    train_pos, temp_pos = next(
        first_splitter.split(
            class_frame,
            groups=class_frame["split_group"],
        )
    )

    temp_frame = class_frame.iloc[temp_pos]
    if temp_frame["split_group"].nunique() < 2:
        raise RuntimeError(
            f"Temporary split has fewer than two groups for {class_name}."
        )

    second_splitter = GroupShuffleSplit(
        n_splits=1,
        train_size=0.50,
        random_state=SEED + 1,
    )
    val_rel_pos, test_rel_pos = next(
        second_splitter.split(
            temp_frame,
            groups=temp_frame["split_group"],
        )
    )

    new_split.loc[class_frame.index[train_pos]] = "train"
    new_split.loc[temp_frame.index[val_rel_pos]] = "val"
    new_split.loc[temp_frame.index[test_rel_pos]] = "test"


for class_name in ["real", "fake"]:
    class_frame = manifest_df.loc[
        manifest_df["label"] == class_name
    ]
    if class_frame.empty:
        raise RuntimeError(f"No {class_name} ROI images were found.")
    assign_grouped_split(class_frame, class_name)

if new_split.isna().any():
    raise RuntimeError("Grouped split assignment left unassigned samples.")

manifest_df["split"] = new_split.astype(str)

# Every split must contain both labels.
split_label_counts = (
    manifest_df.groupby("split")["label"].nunique().to_dict()
)
for split_name in ["train", "val", "test"]:
    if split_label_counts.get(split_name, 0) != 2:
        raise RuntimeError(
            f"{split_name} does not contain both Real and Fake classes."
        )

split_group_sets = {
    split_name: set(
        manifest_df.loc[
            manifest_df["split"] == split_name,
            "split_group",
        ].astype(str)
    )
    for split_name in ["train", "val", "test"]
}

if not split_group_sets["train"].isdisjoint(split_group_sets["val"]):
    raise RuntimeError("Train/Val split-group overlap remains.")
if not split_group_sets["train"].isdisjoint(split_group_sets["test"]):
    raise RuntimeError("Train/Test split-group overlap remains.")
if not split_group_sets["val"].isdisjoint(split_group_sets["test"]):
    raise RuntimeError("Val/Test split-group overlap remains.")

split_assignment_audit = manifest_df[
    [
        "file_name",
        "label",
        "source_video",
        "source_group",
        "split_group",
        "original_split",
        "split",
        "sha256",
        "orijinal_yol",
    ]
].copy()

split_assignment_audit.to_csv(
    ARTIFACT_DIR / "source_group_split_assignment.csv",
    index=False,
)

split_summary = (
    manifest_df.groupby(["label", "split"])
    .agg(
        images=("file_name", "size"),
        source_groups=("source_group", "nunique"),
        leakproof_groups=("split_group", "nunique"),
        source_videos=("source_video", "nunique"),
    )
    .reset_index()
)

print("\nMetadata mapping        : PASS")
print("Hash-aware group build : PASS")
print("Source-group re-split  : PASS")
print("\nFinal in-memory split summary:")
print(split_summary.to_string(index=False))

print("\nIMPORTANT:")
print("Physical Kaş folders were NOT modified.")
print("Training uses the corrected in-memory source-group split above.")


Direct ROI-to-frame metadata match rate: 100.00%
Calculating image SHA-256 values...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Metadata mapping        : PASS
Hash-aware group build : PASS
Source-group re-split  : PASS

Final in-memory split summary:
label split  images  source_groups  leakproof_groups  source_videos
 fake  test      91             45                45             83
 fake train     759            354               354            671
 fake   val     119             44                44            103
 real  test      98             58                58             58
 real train     789            457               457            457
 real   val     106             57                57             57

IMPORTANT:
Physical Kaş folders were NOT modified.
Training uses the corrected in-memory source-group split above.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
# ============================================================
# CELL 6 — STANDARD MANIFEST, ACCOUNTING AND LEAKAGE GATES
# ============================================================

def stable_sample_id(row):
    token = (
        f"{row['label']}|{row['source_video']}|"
        f"{row['frame_index']}|{row['file_name']}|{row['sha256']}"
    )
    return hashlib.sha256(token.encode("utf-8")).hexdigest()[:16]


manifest_df["sample_id"] = manifest_df.apply(stable_sample_id, axis=1)
manifest_df["face_index"] = 0
manifest_df["roi_state"] = "eyebrow_pair"
manifest_df["status"] = "SUCCESS"
manifest_df["skip_reason"] = ""
manifest_df["run_id"] = RUN_ID
manifest_df["output_path"] = manifest_df["path"]

standard_manifest = manifest_df[
    [
        "sample_id",
        "source_video",
        "frame_index",
        "face_index",
        "roi_state",
        "label",
        "split",
        "status",
        "skip_reason",
        "sha256",
        "output_path",
        "run_id",
        "file_name",
        "label_id",
        "orijinal_yol",
        "original_split",
        "source_group",
        "split_group",
    ]
].copy()

if not standard_manifest["sample_id"].is_unique:
    raise RuntimeError("Duplicate sample_id values were generated.")
if standard_manifest["output_path"].isna().any():
    raise RuntimeError("Missing output_path exists in model manifest.")

missing_success_files = [
    path
    for path in standard_manifest.loc[
        standard_manifest["status"] == "SUCCESS",
        "output_path",
    ]
    if not Path(path).exists()
]
if missing_success_files:
    raise RuntimeError(
        f"SUCCESS manifest files are missing on disk. Examples: {missing_success_files[:10]}"
    )

total_inputs = len(standard_manifest)
success_count = int((standard_manifest["status"] == "SUCCESS").sum())
skipped_count = int((standard_manifest["status"] == "SKIPPED").sum())
error_count = int((standard_manifest["status"] == "ERROR").sum())

if total_inputs != success_count + skipped_count + error_count:
    raise RuntimeError("Accounting equality failed.")

MANIFEST_PATH = ARTIFACT_DIR / "model_manifest.csv"
standard_manifest.to_csv(MANIFEST_PATH, index=False)

accounting = {
    "total_inputs": total_inputs,
    "success": success_count,
    "skipped": skipped_count,
    "error": error_count,
    "accounting_equality_pass": True,
}
with open(ARTIFACT_DIR / "accounting_summary.json", "w", encoding="utf-8") as f:
    json.dump(accounting, f, indent=2, ensure_ascii=False)


def pairwise_overlap(split_sets):
    return {
        "train_val": sorted(split_sets["train"] & split_sets["val"]),
        "train_test": sorted(split_sets["train"] & split_sets["test"]),
        "val_test": sorted(split_sets["val"] & split_sets["test"]),
    }


# Exact source-video isolation is checked within each class.
exact_source_overlaps_by_class = {}
for label_name in ["real", "fake"]:
    class_split_sets = {
        split_name: set(
            standard_manifest.loc[
                (standard_manifest["label"] == label_name)
                & (standard_manifest["split"] == split_name),
                "source_video",
            ].astype(str)
        )
        for split_name in ["train", "val", "test"]
    }
    exact_source_overlaps_by_class[label_name] = pairwise_overlap(
        class_split_sets
    )

group_split_sets = {
    split_name: set(
        standard_manifest.loc[
            standard_manifest["split"] == split_name,
            "split_group",
        ].astype(str)
    )
    for split_name in ["train", "val", "test"]
}
group_overlaps = pairwise_overlap(group_split_sets)

hash_split_counts = standard_manifest.groupby("sha256")["split"].nunique()
cross_split_duplicate_hashes = hash_split_counts[hash_split_counts > 1]

exact_source_pass = all(
    len(values) == 0
    for class_report in exact_source_overlaps_by_class.values()
    for values in class_report.values()
)
group_isolation_pass = all(
    len(values) == 0
    for values in group_overlaps.values()
)
content_isolation_pass = cross_split_duplicate_hashes.empty

class_presence = (
    standard_manifest.groupby("split")["label"].nunique().to_dict()
)
class_presence_pass = all(
    class_presence.get(split_name, 0) == 2
    for split_name in ["train", "val", "test"]
)

leakage_pass = (
    exact_source_pass
    and group_isolation_pass
    and content_isolation_pass
    and class_presence_pass
)

leakage_report = {
    "policy": CONFIG["split_policy"],
    "exact_source_video_overlap_by_class": exact_source_overlaps_by_class,
    "source_group_overlap": group_overlaps,
    "cross_split_duplicate_hash_count": int(
        len(cross_split_duplicate_hashes)
    ),
    "class_presence": {
        key: int(value)
        for key, value in class_presence.items()
    },
    "exact_source_pass": bool(exact_source_pass),
    "group_isolation_pass": bool(group_isolation_pass),
    "content_isolation_pass": bool(content_isolation_pass),
    "class_presence_pass": bool(class_presence_pass),
    "leakage_pass": bool(leakage_pass),
    "unique_split_groups": {
        split_name: int(len(group_values))
        for split_name, group_values in group_split_sets.items()
    },
}

with open(ARTIFACT_DIR / "leakage_check.json", "w", encoding="utf-8") as f:
    json.dump(leakage_report, f, indent=2, ensure_ascii=False)

if not exact_source_pass:
    raise RuntimeError(
        "DATA LEAKAGE GATE FAILED: exact source_video occurs in more than one "
        f"split within the same class.\n{exact_source_overlaps_by_class}"
    )
if not group_isolation_pass:
    raise RuntimeError(
        "DATA LEAKAGE GATE FAILED: split-group overlap remains.\n"
        f"{group_overlaps}"
    )
if not content_isolation_pass:
    examples = cross_split_duplicate_hashes.index[:20].tolist()
    raise RuntimeError(
        "DATA LEAKAGE GATE FAILED: identical image content exists in more than "
        "one split after hash-aware grouping.\n"
        f"Duplicate hash count: {len(cross_split_duplicate_hashes)}\n"
        f"Examples: {examples}"
    )
if not class_presence_pass:
    raise RuntimeError(
        f"At least one split is missing a class: {class_presence}"
    )

print("Accounting equality     : PASS")
print("Exact source isolation  : PASS")
print("Source-group isolation  : PASS")
print("Content leakage         : PASS")
print("Class presence          : PASS")


Accounting equality     : PASS
Exact source isolation  : PASS
Source-group isolation  : PASS
Content leakage         : PASS
Class presence          : PASS


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
# ============================================================
# CELL 7 — TF.DATA PIPELINE + EYEBROW-SAFE AUGMENTATION
# ============================================================

from tensorflow.keras.applications.densenet import preprocess_input

IMAGE_HEIGHT, IMAGE_WIDTH = CONFIG["image_size"]
BATCH_SIZE = CONFIG["batch_size"]
AUTOTUNE = tf.data.AUTOTUNE

train_df = standard_manifest.loc[
    standard_manifest["split"] == "train"
].reset_index(drop=True)
val_df = standard_manifest.loc[
    standard_manifest["split"] == "val"
].reset_index(drop=True)
test_df = standard_manifest.loc[
    standard_manifest["split"] == "test"
].reset_index(drop=True)

if min(len(train_df), len(val_df), len(test_df)) <= 0:
    raise RuntimeError("Train/Val/Test contains an empty split.")

data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(
            0.025,
            fill_mode="reflect",
            seed=SEED,
        ),
        tf.keras.layers.RandomZoom(
            height_factor=(-0.05, 0.05),
            width_factor=(-0.05, 0.05),
            fill_mode="reflect",
            seed=SEED,
        ),
        tf.keras.layers.RandomContrast(0.10, seed=SEED),
    ],
    name="eyebrow_data_augmentation",
)


def decode_and_letterbox(path, label, training=False):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False,
    )
    image.set_shape([None, None, 3])
    image = tf.cast(image, tf.float32)

    # Preserve the eyebrow band's aspect ratio.
    image = tf.image.resize_with_pad(
        image,
        target_height=IMAGE_HEIGHT,
        target_width=IMAGE_WIDTH,
        method="bilinear",
    )

    if training:
        image = data_augmentation(image, training=True)

    # Augmentation can slightly leave the original pixel range.
    image = tf.clip_by_value(image, 0.0, 255.0)
    image = preprocess_input(image)

    # Dense(1) produces shape (batch, 1), so labels are explicitly (1,).
    label = tf.reshape(tf.cast(label, tf.float32), [1])
    return image, label


def make_dataset(frame, training):
    paths = frame["output_path"].astype(str).to_numpy()
    labels = frame["label_id"].astype(np.float32).to_numpy()

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(frame),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.map(
        lambda p, y: decode_and_letterbox(p, y, training=training),
        num_parallel_calls=AUTOTUNE,
        # Keep map ordering deterministic; randomness comes from seeded layers.
        deterministic=True,
    )

    dataset = dataset.batch(BATCH_SIZE, drop_remainder=False)
    dataset = dataset.prefetch(AUTOTUNE)
    return dataset


train_dataset = make_dataset(train_df, training=True)
val_dataset = make_dataset(val_df, training=False)
test_dataset = make_dataset(test_df, training=False)

train_batches = int(tf.data.experimental.cardinality(train_dataset).numpy())
val_batches = int(tf.data.experimental.cardinality(val_dataset).numpy())
test_batches = int(tf.data.experimental.cardinality(test_dataset).numpy())

if min(train_batches, val_batches, test_batches) <= 0:
    raise RuntimeError(
        f"Invalid dataset cardinality: train={train_batches}, "
        f"val={val_batches}, test={test_batches}"
    )

sample_images, sample_labels = next(iter(val_dataset))

print("Dataset batches          :", train_batches, val_batches, test_batches)
print("Validation image shape   :", sample_images.shape)
print("Validation label shape   :", sample_labels.shape)
print("Image dtype              :", sample_images.dtype)
print("Label dtype              :", sample_labels.dtype)

if tuple(sample_images.shape[1:]) != (IMAGE_HEIGHT, IMAGE_WIDTH, 3):
    raise RuntimeError(f"Unexpected image batch shape: {sample_images.shape}")
if sample_labels.shape.rank != 2 or sample_labels.shape[-1] != 1:
    raise RuntimeError(f"Unexpected binary-label shape: {sample_labels.shape}")
if not bool(tf.reduce_all(tf.math.is_finite(sample_images)).numpy()):
    raise FloatingPointError("NaN/Inf exists in preprocessed sample images.")
if not bool(
    tf.reduce_all(
        tf.logical_or(
            tf.equal(sample_labels, 0.0),
            tf.equal(sample_labels, 1.0),
        )
    ).numpy()
):
    raise RuntimeError("Labels outside {0,1} were found.")

print("TF.DATA PIPELINE: PASS")


Dataset batches          : 97 15 12
Validation image shape   : (16, 224, 224, 3)
Validation label shape   : (16, 1)
Image dtype              : <dtype: 'float32'>
Label dtype              : <dtype: 'float32'>


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


TF.DATA PIPELINE: PASS


In [8]:
# ============================================================
# CELL 8 — BUILD IMAGENET-PRETRAINED DENSENET121
# ============================================================

from tensorflow.keras import Model
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import mixed_precision

if GPUS:
    mixed_precision.set_global_policy("mixed_float16")
else:
    mixed_precision.set_global_policy("float32")

print("Mixed precision policy:", mixed_precision.global_policy())


def model_metrics():
    return [
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc", curve="ROC"),
        tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ]


def compile_binary_model(model_object, learning_rate):
    model_object.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=model_metrics(),
    )
    return model_object


def build_frozen_model():
    base_model = DenseNet121(
        weights=CONFIG["pretrained_weights"],
        include_top=False,
        input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3),
    )
    base_model.trainable = False

    inputs = tf.keras.Input(
        shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3),
        name="eyebrow_roi_input",
    )
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)
    x = tf.keras.layers.Dropout(
        CONFIG["dropout_rate"],
        name="classification_dropout",
    )(x)
    outputs = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        dtype="float32",
        name="fake_probability",
    )(x)

    model_object = Model(
        inputs=inputs,
        outputs=outputs,
        name="DenseNet121_Eyebrow_Deepfake",
    )
    return compile_binary_model(
        model_object,
        CONFIG["frozen_learning_rate"],
    )


model = build_frozen_model()
base_model = model.get_layer("densenet121")

if base_model.trainable:
    raise RuntimeError("DenseNet121 backbone should be frozen in stage 1.")
if tuple(model.output_shape) != (None, 1):
    raise RuntimeError(f"Unexpected model output shape: {model.output_shape}")

summary_lines = []
model.summary(print_fn=summary_lines.append)
(ARTIFACT_DIR / "model_summary.txt").write_text(
    "\n".join(summary_lines),
    encoding="utf-8",
)

print(model.name)
print("Total parameters    :", model.count_params())
print("Backbone trainable  :", base_model.trainable)


Mixed precision policy: <DTypePolicy "mixed_float16">
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


DenseNet121_Eyebrow_Deepfake
Total parameters    : 7038529
Backbone trainable  : False


In [9]:
# ============================================================
# CELL 9 — QUALITY GATES: SCHEMA, SPLIT, SMOKE, NUMERICAL, CHECKPOINT
# ============================================================

quality_gates = {
    "schema_test": False,
    "split_test": False,
    "smoke_test": False,
    "numerical_test": False,
    "checkpoint_test": False,
}

required_manifest_columns = {
    "sample_id",
    "source_video",
    "frame_index",
    "face_index",
    "roi_state",
    "label",
    "split",
    "status",
    "skip_reason",
    "sha256",
    "output_path",
    "run_id",
    "source_group",
    "split_group",
}

if not required_manifest_columns.issubset(standard_manifest.columns):
    missing = sorted(required_manifest_columns - set(standard_manifest.columns))
    raise RuntimeError(f"Schema gate failed. Missing columns: {missing}")
if not standard_manifest["sample_id"].is_unique:
    raise RuntimeError("Schema gate failed: sample_id is not unique.")
quality_gates["schema_test"] = True

if not leakage_pass:
    raise RuntimeError("Split/leakage gate failed.")
quality_gates["split_test"] = True

# ------------------------------------------------------------
# Two-batch forward/backward smoke test.
# IMPORTANT: return_dict=True avoids scalar/list differences across Keras versions.
# ------------------------------------------------------------
smoke_model = tf.keras.models.clone_model(model)
smoke_model.set_weights(model.get_weights())
compile_binary_model(
    smoke_model,
    CONFIG["frozen_learning_rate"],
)

smoke_losses = []

for batch_index, (x_batch, y_batch) in enumerate(
    train_dataset.take(2),
    start=1,
):
    if y_batch.shape.rank != 2 or y_batch.shape[-1] != 1:
        raise RuntimeError(
            f"Smoke-test label shape is incompatible: {y_batch.shape}"
        )

    batch_result = smoke_model.train_on_batch(
        x_batch,
        y_batch,
        return_dict=True,
    )

    if not isinstance(batch_result, dict) or "loss" not in batch_result:
        raise RuntimeError(
            f"Unexpected train_on_batch result: {batch_result!r}"
        )

    loss_array = np.asarray(batch_result["loss"], dtype=np.float64).reshape(-1)
    if loss_array.size != 1:
        raise RuntimeError(
            f"Smoke loss is not scalar: {batch_result['loss']!r}"
        )

    loss_value = float(loss_array[0])
    if not np.isfinite(loss_value):
        raise FloatingPointError(
            f"Non-finite smoke loss at batch {batch_index}: {loss_value}"
        )

    def assert_finite_numeric(value, name):
        if isinstance(value, dict):
            for sub_name, sub_value in value.items():
                assert_finite_numeric(
                    sub_value,
                    f"{name}.{sub_name}",
                )
            return

        array = np.asarray(value, dtype=np.float64).reshape(-1)
        if array.size == 0 or not np.isfinite(array).all():
            raise FloatingPointError(
                f"Non-finite numeric value: {name}={value!r}"
            )

    for metric_name, metric_value in batch_result.items():
        assert_finite_numeric(
            metric_value,
            f"smoke.{metric_name}",
        )

    smoke_losses.append(loss_value)
    print(f"Smoke batch {batch_index}: loss={loss_value:.6f}")

if len(smoke_losses) != 2:
    raise RuntimeError(
        f"Smoke test expected 2 batches but processed {len(smoke_losses)}."
    )

quality_gates["smoke_test"] = True
quality_gates["numerical_test"] = True

# ------------------------------------------------------------
# Complete-model .keras save/load integrity test.
# ------------------------------------------------------------
smoke_checkpoint = CHECKPOINT_DIR / "_smoke_test.keras"
smoke_temp = CHECKPOINT_DIR / "_smoke_test.tmp.keras"

for path in [smoke_checkpoint, smoke_temp]:
    if path.exists():
        path.unlink()

test_x, _ = next(iter(val_dataset))
reference_predictions = smoke_model(
    test_x[:2],
    training=False,
).numpy()

smoke_model.save(smoke_temp)
if not smoke_temp.exists() or smoke_temp.stat().st_size <= 0:
    raise RuntimeError("Temporary smoke checkpoint was not written.")

loaded_smoke_model = tf.keras.models.load_model(smoke_temp)
loaded_predictions = loaded_smoke_model(
    test_x[:2],
    training=False,
).numpy()

np.testing.assert_allclose(
    reference_predictions,
    loaded_predictions,
    rtol=1e-5,
    atol=1e-6,
)

if tuple(loaded_smoke_model.output_shape) != (None, 1):
    raise RuntimeError(
        f"Loaded smoke model output shape is invalid: "
        f"{loaded_smoke_model.output_shape}"
    )

os.replace(smoke_temp, smoke_checkpoint)
if not smoke_checkpoint.exists():
    raise RuntimeError("Atomic smoke checkpoint publish failed.")

smoke_checkpoint.unlink()
del smoke_model, loaded_smoke_model, reference_predictions, loaded_predictions
gc.collect()

quality_gates["checkpoint_test"] = True

with open(METRICS_DIR / "quality_gates.json", "w", encoding="utf-8") as f:
    json.dump(quality_gates, f, indent=2)

print("\nQUALITY GATES")
for key, value in quality_gates.items():
    print(f"{key:20}: {'PASS' if value else 'FAIL'}")

if not all(quality_gates.values()):
    raise RuntimeError(f"Quality gates failed: {quality_gates}")

print("ALL PRE-TRAINING QUALITY GATES: PASS")


Smoke batch 1: loss=0.858280
Smoke batch 2: loss=0.763165

QUALITY GATES
schema_test         : PASS
split_test          : PASS
smoke_test          : PASS
numerical_test      : PASS
checkpoint_test     : PASS
ALL PRE-TRAINING QUALITY GATES: PASS


In [10]:
# ============================================================
# CELL 10 — ATOMIC CHECKPOINT CALLBACKS + RESUME-SAFE LOGGING
# ============================================================

BEST_MODEL_PATH = CHECKPOINT_DIR / "best.keras"
LAST_MODEL_PATH = CHECKPOINT_DIR / "last.keras"
STATE_PATH = CHECKPOINT_DIR / "training_state.json"


def atomic_json_write(path, payload):
    path = Path(path)
    temp_path = path.with_suffix(path.suffix + ".tmp")
    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    os.replace(temp_path, path)


def scalar_float(value, name="value"):
    array = np.asarray(value, dtype=np.float64).reshape(-1)
    if array.size != 1:
        raise RuntimeError(f"{name} is not scalar: {value!r}")
    result = float(array[0])
    if not np.isfinite(result):
        raise FloatingPointError(f"{name} is NaN/Inf: {value!r}")
    return result


class AtomicModelCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, stage_name, monitor="val_auc", mode="max"):
        super().__init__()
        self.stage_name = str(stage_name)
        self.monitor = monitor
        self.mode = mode

        if mode == "max":
            self.best_value = -np.inf
        elif mode == "min":
            self.best_value = np.inf
        else:
            raise ValueError("mode must be 'max' or 'min'.")

        if STATE_PATH.exists():
            state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
            if monitor in state:
                self.best_value = float(state[monitor])

    def _improved(self, current):
        return (
            current > self.best_value
            if self.mode == "max"
            else current < self.best_value
        )

    def _atomic_save_model(self, target):
        target = Path(target)
        temp = target.parent / f"{target.stem}.tmp.keras"

        if temp.exists():
            temp.unlink()

        # Native .keras format already stores architecture, weights,
        # compile information and optimizer state.
        self.model.save(temp)

        if not temp.exists() or temp.stat().st_size <= 0:
            raise RuntimeError(f"Checkpoint temp file was not written: {temp}")

        verification_model = tf.keras.models.load_model(temp)
        if tuple(verification_model.output_shape) != (None, 1):
            raise RuntimeError(
                f"Checkpoint verification failed: "
                f"{verification_model.output_shape}"
            )
        del verification_model

        os.replace(temp, target)

        if not target.exists() or target.stat().st_size <= 0:
            raise RuntimeError(f"Checkpoint publish failed: {target}")

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if self.monitor not in logs:
            raise RuntimeError(
                f"Monitored metric {self.monitor} is missing from logs: "
                f"{sorted(logs.keys())}"
            )

        current = scalar_float(
            logs[self.monitor],
            name=self.monitor,
        )

        self._atomic_save_model(LAST_MODEL_PATH)

        if self._improved(current):
            self.best_value = current
            self._atomic_save_model(BEST_MODEL_PATH)

        state = {
            "run_id": RUN_ID,
            "stage": self.stage_name,
            "completed_epoch": int(epoch + 1),
            self.monitor: float(self.best_value),
            "last_epoch_metric": float(current),
            "seed": SEED,
            "tensorflow_random_seed": SEED,
            "numpy_random_seed": SEED,
            "python_random_seed": SEED,
        }
        atomic_json_write(STATE_PATH, state)


def ensure_finite_numeric(value, name="value"):
    if isinstance(value, dict):
        for sub_name, sub_value in value.items():
            ensure_finite_numeric(
                sub_value,
                name=f"{name}.{sub_name}",
            )
        return

    array = np.asarray(value, dtype=np.float64).reshape(-1)
    if array.size == 0 or not np.isfinite(array).all():
        raise FloatingPointError(
            f"{name} contains NaN/Inf: {value!r}"
        )


class FiniteMetricsGuard(tf.keras.callbacks.Callback):
    def on_train_batch_end(self, batch, logs=None):
        logs = logs or {}
        for key, value in logs.items():
            ensure_finite_numeric(
                value,
                name=f"train_batch_{key}",
            )

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        for key, value in logs.items():
            ensure_finite_numeric(
                value,
                name=f"epoch_{key}",
            )


def build_callbacks(stage_name):
    return [
        tf.keras.callbacks.TerminateOnNaN(),
        AtomicModelCheckpoint(
            stage_name=stage_name,
            monitor=CONFIG["model_selection_metric"],
            mode=CONFIG["model_selection_mode"],
        ),
        FiniteMetricsGuard(),
        tf.keras.callbacks.CSVLogger(
            str(LOG_DIR / f"{stage_name}_training_log.csv"),
            # Append is important when BackupAndRestore resumes a run.
            append=True,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.2,
            patience=CONFIG["reduce_lr_patience"],
            min_lr=1e-7,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor=CONFIG["model_selection_metric"],
            mode=CONFIG["model_selection_mode"],
            patience=CONFIG["early_stopping_patience"],
            restore_best_weights=False,
            verbose=1,
        ),
        tf.keras.callbacks.BackupAndRestore(
            backup_dir=str(CHECKPOINT_DIR / f"{stage_name}_backup"),
            save_freq="epoch",
            delete_checkpoint=False,
        ),
    ]


def read_stage_history(log_path, stage_name):
    log_path = Path(log_path)
    if not log_path.exists() or log_path.stat().st_size <= 0:
        raise RuntimeError(f"Training CSV log is missing/empty: {log_path}")

    frame = pd.read_csv(log_path)
    if frame.empty:
        raise RuntimeError(f"Training CSV log has no rows: {log_path}")
    frame = frame.copy()

    # CSVLogger normally writes an epoch column. If a future Keras version
    # omits it, fall back to row order instead of failing the run.
    if "epoch" in frame.columns:
        frame["epoch"] = (
            pd.to_numeric(
                frame["epoch"],
                errors="raise",
            ).astype(int)
            + 1
        )
    else:
        frame.insert(
            0,
            "epoch",
            np.arange(1, len(frame) + 1, dtype=int),
        )
    frame = (
        frame.drop_duplicates(subset=["epoch"], keep="last")
        .sort_values("epoch")
        .reset_index(drop=True)
    )
    frame["stage"] = stage_name
    return frame


print("Checkpoint target:", CHECKPOINT_DIR)


Checkpoint target: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/DenseNet121_Kas_Results_20260808_1335_eyebrow_densenet121_seed42/checkpoints


In [11]:
# ============================================================
# CELL 11 — STAGE 1: FROZEN BACKBONE TRAINING
# ============================================================

FROZEN_LOG_PATH = LOG_DIR / "frozen_training_log.csv"

frozen_fit_history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=CONFIG["frozen_epochs"],
    callbacks=build_callbacks("frozen"),
    verbose=1,
)

if not BEST_MODEL_PATH.exists():
    raise RuntimeError("best.keras was not created during frozen training.")
if not LAST_MODEL_PATH.exists():
    raise RuntimeError("last.keras was not created during frozen training.")
if not STATE_PATH.exists():
    raise RuntimeError("training_state.json was not created.")

# Read the CSV log rather than relying only on the current History object.
# This stays correct when BackupAndRestore resumes after an interruption.
frozen_history_df = read_stage_history(
    FROZEN_LOG_PATH,
    "frozen",
)

frozen_history_df.to_csv(
    METRICS_DIR / "frozen_training_history.csv",
    index=False,
)

FROZEN_EPOCHS_COMPLETED = int(frozen_history_df["epoch"].max())

print("Frozen-stage epochs completed:", FROZEN_EPOCHS_COMPLETED)
print("Best checkpoint:", BEST_MODEL_PATH)


Epoch 1/12
97/97 ━━━━━━━━━━━━━━━━━━━━ 65s 510ms/step - accuracy: 0.5000 - auc: 0.5049 - loss: 0.7485 - pr_auc: 0.5007 - precision: 0.4897 - recall: 0.4677 - val_accuracy: 0.4756 - val_auc: 0.4916 - val_loss: 0.7319 - val_pr_auc: 0.5491 - val_precision: 0.5263 - val_recall: 0.0840 - learning_rate: 0.0010
Epoch 2/12
97/97 ━━━━━━━━━━━━━━━━━━━━ 46s 478ms/step - accuracy: 0.5136 - auc: 0.5274 - loss: 0.7237 - pr_auc: 0.5150 - precision: 0.5043 - recall: 0.4598 - val_accuracy: 0.5289 - val_auc: 0.5145 - val_loss: 0.6959 - val_pr_auc: 0.5562 - val_precision: 0.5419 - val_recall: 0.7059 - learning_rate: 0.0010
Epoch 3/12
97/97 ━━━━━━━━━━━━━━━━━━━━ 48s 493ms/step - accuracy: 0.5401 - auc: 0.5641 - loss: 0.7013 - pr_auc: 0.5405 - precision: 0.5298 - recall: 0.5507 - val_accuracy: 0.4800 - val_auc: 0.5218 - val_loss: 0.7161 - val_pr_auc: 0.5529 - val_precision: 0.5278 - val_recall: 0.1597 - learning_rate: 0.0010
Epoch 4/12
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 358ms/step - accuracy: 0.5660 - auc: 0.5817 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
# ============================================================
# CELL 12 — STAGE 2: PARTIAL DENSENET121 FINE-TUNING
# ============================================================

if not BEST_MODEL_PATH.exists():
    raise RuntimeError("Best frozen-stage checkpoint is missing.")

model = tf.keras.models.load_model(BEST_MODEL_PATH)
base_model = model.get_layer("densenet121")
base_model.trainable = True

for layer in base_model.layers:
    layer.trainable = False

unfrozen_layers = []

for layer in base_model.layers:
    is_final_block = layer.name.startswith("conv5_block")
    is_batch_norm = isinstance(
        layer,
        tf.keras.layers.BatchNormalization,
    )

    if is_final_block and not is_batch_norm:
        layer.trainable = True
        unfrozen_layers.append(layer.name)

if not unfrozen_layers:
    raise RuntimeError(
        "No conv5_block DenseNet121 layers were found for fine-tuning."
    )

if any(
    layer.trainable
    for layer in base_model.layers
    if isinstance(layer, tf.keras.layers.BatchNormalization)
):
    raise RuntimeError("BatchNormalization layers must remain frozen.")

compile_binary_model(
    model,
    CONFIG["finetune_learning_rate"],
)

initial_epoch = int(FROZEN_EPOCHS_COMPLETED)
final_epoch = initial_epoch + int(CONFIG["finetune_epochs"])

FINETUNE_LOG_PATH = LOG_DIR / "finetune_training_log.csv"

finetune_fit_history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    initial_epoch=initial_epoch,
    epochs=final_epoch,
    callbacks=build_callbacks("finetune"),
    verbose=1,
)

if not BEST_MODEL_PATH.exists() or not LAST_MODEL_PATH.exists():
    raise RuntimeError("Fine-tuning checkpoints are missing.")

finetune_history_df = read_stage_history(
    FINETUNE_LOG_PATH,
    "finetune",
)

finetune_history_df.to_csv(
    METRICS_DIR / "finetune_training_history.csv",
    index=False,
)

combined_history_df = pd.concat(
    [frozen_history_df, finetune_history_df],
    ignore_index=True,
)

combined_history_df = (
    combined_history_df
    .drop_duplicates(subset=["epoch"], keep="last")
    .sort_values("epoch")
    .reset_index(drop=True)
)

combined_history_df.to_csv(
    METRICS_DIR / "combined_training_history.csv",
    index=False,
)

print("Fine-tuning logged epochs:", len(finetune_history_df))
print("Global final epoch index:", int(combined_history_df["epoch"].max()))
print("Unfrozen DenseNet layers:", len(unfrozen_layers))


Epoch 13/22
97/97 ━━━━━━━━━━━━━━━━━━━━ 71s 533ms/step - accuracy: 0.5917 - auc: 0.6241 - loss: 0.6687 - pr_auc: 0.6114 - precision: 0.5876 - recall: 0.5613 - val_accuracy: 0.5422 - val_auc: 0.5505 - val_loss: 0.6846 - val_pr_auc: 0.5796 - val_precision: 0.5548 - val_recall: 0.6807 - learning_rate: 1.0000e-05
Epoch 14/22
97/97 ━━━━━━━━━━━━━━━━━━━━ 50s 520ms/step - accuracy: 0.5911 - auc: 0.6207 - loss: 0.6702 - pr_auc: 0.6029 - precision: 0.5827 - recall: 0.5850 - val_accuracy: 0.5378 - val_auc: 0.5562 - val_loss: 0.6844 - val_pr_auc: 0.5911 - val_precision: 0.5581 - val_recall: 0.6050 - learning_rate: 1.0000e-05
Epoch 15/22
97/97 ━━━━━━━━━━━━━━━━━━━━ 55s 568ms/step - accuracy: 0.6001 - auc: 0.6285 - loss: 0.6663 - pr_auc: 0.6060 - precision: 0.5916 - recall: 0.5955 - val_accuracy: 0.5378 - val_auc: 0.5611 - val_loss: 0.6875 - val_pr_auc: 0.5909 - val_precision: 0.5743 - val_recall: 0.4874 - learning_rate: 1.0000e-05
Epoch 16/22
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 407ms/step - accuracy: 0.609

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [13]:
# ============================================================
# CELL 13 — VALIDATION-ONLY THRESHOLD + FINAL TEST EVALUATION
# ============================================================

if not BEST_MODEL_PATH.exists():
    raise RuntimeError("Global best checkpoint is missing.")

best_model = tf.keras.models.load_model(BEST_MODEL_PATH)


def predict_dataset(model_object, dataset):
    labels = []
    probabilities = []

    for x_batch, y_batch in dataset:
        probs = model_object(
            x_batch,
            training=False,
        ).numpy().reshape(-1)

        labels.extend(
            y_batch.numpy().astype(int).reshape(-1).tolist()
        )
        probabilities.extend(
            probs.astype(float).tolist()
        )

    labels = np.asarray(labels, dtype=np.int32)
    probabilities = np.asarray(probabilities, dtype=np.float64)

    if len(labels) == 0:
        raise RuntimeError("Prediction dataset produced zero samples.")
    if not np.isfinite(probabilities).all():
        raise FloatingPointError("Non-finite prediction probability found.")
    if not (
        (probabilities >= 0.0) & (probabilities <= 1.0)
    ).all():
        raise RuntimeError("Predicted probability is outside [0,1].")

    return labels, probabilities


# Threshold selection uses validation only.
val_labels, val_probabilities = predict_dataset(
    best_model,
    val_dataset,
)

if np.unique(val_labels).size != 2:
    raise RuntimeError(
        f"Validation set must contain both classes: {np.unique(val_labels)}"
    )

val_fpr, val_tpr, val_thresholds = roc_curve(
    val_labels,
    val_probabilities,
)

finite_mask = np.isfinite(val_thresholds)
if int(finite_mask.sum()) == 0:
    raise RuntimeError("Validation ROC returned no finite threshold.")

youden_j = val_tpr[finite_mask] - val_fpr[finite_mask]
best_index = int(np.argmax(youden_j))
SELECTED_THRESHOLD = float(
    val_thresholds[finite_mask][best_index]
)

if not np.isfinite(SELECTED_THRESHOLD):
    raise RuntimeError("Selected threshold is not finite.")

threshold_payload = {
    "selection_dataset": "validation",
    "method": "Youden J",
    "selected_threshold": SELECTED_THRESHOLD,
    "youden_j": float(youden_j[best_index]),
}

with open(
    METRICS_DIR / "validation_threshold.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(threshold_payload, f, indent=2)

# Test is touched only after model and threshold selection are finished.
test_labels, test_probabilities = predict_dataset(
    best_model,
    test_dataset,
)

if np.unique(test_labels).size != 2:
    raise RuntimeError(
        f"Test set must contain both classes: {np.unique(test_labels)}"
    )

test_predictions = (
    test_probabilities >= SELECTED_THRESHOLD
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    test_labels,
    test_predictions,
    labels=[0, 1],
).ravel()

specificity = (
    float(tn / (tn + fp))
    if (tn + fp) > 0
    else float("nan")
)

image_metrics = {
    "accuracy": float(
        accuracy_score(test_labels, test_predictions)
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(test_labels, test_predictions)
    ),
    "precision": float(
        precision_score(
            test_labels,
            test_predictions,
            zero_division=0,
        )
    ),
    "recall_sensitivity": float(
        recall_score(
            test_labels,
            test_predictions,
            zero_division=0,
        )
    ),
    "specificity": specificity,
    "f1": float(
        f1_score(
            test_labels,
            test_predictions,
            zero_division=0,
        )
    ),
    "roc_auc": float(
        roc_auc_score(test_labels, test_probabilities)
    ),
    "pr_auc": float(
        average_precision_score(
            test_labels,
            test_probabilities,
        )
    ),
    "mcc": float(
        matthews_corrcoef(
            test_labels,
            test_predictions,
        )
    ),
    "threshold": float(SELECTED_THRESHOLD),
    "n_test_images": int(len(test_labels)),
}

if not all(
    np.isfinite(value)
    for key, value in image_metrics.items()
    if key != "n_test_images"
):
    raise FloatingPointError(
        f"Non-finite image-level metric found: {image_metrics}"
    )

pd.DataFrame([image_metrics]).to_csv(
    METRICS_DIR / "final_image_level_metrics.csv",
    index=False,
)

classification_report_df = pd.DataFrame(
    classification_report(
        test_labels,
        test_predictions,
        labels=[0, 1],
        target_names=["real", "fake"],
        output_dict=True,
        zero_division=0,
    )
).T

classification_report_df.to_csv(
    METRICS_DIR / "classification_report.csv",
    index=True,
)

# test_dataset is not shuffled, so it preserves test_df order.
prediction_df = test_df[
    [
        "sample_id",
        "source_video",
        "source_group",
        "split_group",
        "frame_index",
        "label",
        "output_path",
    ]
].copy()

if len(prediction_df) != len(test_labels):
    raise RuntimeError(
        "Prediction row count differs from test manifest row count."
    )

prediction_df["true_label_id"] = test_labels
prediction_df["fake_probability"] = test_probabilities
prediction_df["predicted_label_id"] = test_predictions
prediction_df["predicted_label"] = np.where(
    test_predictions == 1,
    "fake",
    "real",
)
prediction_df["source_key"] = (
    prediction_df["label"].astype(str)
    + "::"
    + prediction_df["source_video"].astype(str)
)

prediction_df.to_csv(
    PREDICTION_DIR / "test_image_predictions.csv",
    index=False,
)

# Video/source-level aggregation.
source_label_check = (
    prediction_df.groupby("source_key")["true_label_id"].nunique()
)
if (source_label_check > 1).any():
    raise RuntimeError(
        "A source_key has conflicting class labels in the test set."
    )

source_predictions = (
    prediction_df.groupby(
        ["source_key", "label", "source_video"],
        as_index=False,
    )
    .agg(
        true_label_id=("true_label_id", "first"),
        fake_probability=("fake_probability", "mean"),
        frame_count=("fake_probability", "size"),
    )
)

source_predictions["predicted_label_id"] = (
    source_predictions["fake_probability"] >= SELECTED_THRESHOLD
).astype(int)

source_predictions["predicted_label"] = np.where(
    source_predictions["predicted_label_id"] == 1,
    "fake",
    "real",
)

if source_predictions["true_label_id"].nunique() != 2:
    raise RuntimeError("Source-level test aggregation lost one class.")

source_metrics = {
    "accuracy": float(
        accuracy_score(
            source_predictions["true_label_id"],
            source_predictions["predicted_label_id"],
        )
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(
            source_predictions["true_label_id"],
            source_predictions["predicted_label_id"],
        )
    ),
    "precision": float(
        precision_score(
            source_predictions["true_label_id"],
            source_predictions["predicted_label_id"],
            zero_division=0,
        )
    ),
    "recall": float(
        recall_score(
            source_predictions["true_label_id"],
            source_predictions["predicted_label_id"],
            zero_division=0,
        )
    ),
    "f1": float(
        f1_score(
            source_predictions["true_label_id"],
            source_predictions["predicted_label_id"],
            zero_division=0,
        )
    ),
    "roc_auc": float(
        roc_auc_score(
            source_predictions["true_label_id"],
            source_predictions["fake_probability"],
        )
    ),
    "pr_auc": float(
        average_precision_score(
            source_predictions["true_label_id"],
            source_predictions["fake_probability"],
        )
    ),
    "n_test_sources": int(len(source_predictions)),
    "threshold": float(SELECTED_THRESHOLD),
}

if not all(
    np.isfinite(value)
    for key, value in source_metrics.items()
    if key != "n_test_sources"
):
    raise FloatingPointError(
        f"Non-finite source-level metric found: {source_metrics}"
    )

pd.DataFrame([source_metrics]).to_csv(
    METRICS_DIR / "final_source_level_metrics.csv",
    index=False,
)

source_predictions.to_csv(
    PREDICTION_DIR / "test_source_predictions.csv",
    index=False,
)

print("Selected validation threshold:", SELECTED_THRESHOLD)
print("\nImage-level test metrics:")
for key, value in image_metrics.items():
    print(f"{key:22}: {value}")


Selected validation threshold: 0.5567432641983032

Image-level test metrics:
accuracy              : 0.5026455026455027
balanced_accuracy     : 0.49058084772370486
precision             : 0.45454545454545453
recall_sensitivity    : 0.16483516483516483
specificity           : 0.8163265306122449
f1                    : 0.24193548387096775
roc_auc               : 0.554047992823503
pr_auc                : 0.5292930068489279
mcc                   : -0.02479455273924247
threshold             : 0.5567432641983032
n_test_images         : 189


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [14]:
# ============================================================
# CELL 14 — ENGLISH, >=600px FIGURES
# ============================================================

def save_figure(fig, path):
    path = Path(path)
    fig.tight_layout()
    fig.savefig(
        path,
        dpi=150,
        bbox_inches="tight",
        format="png",
    )
    plt.close(fig)

    if not path.exists() or path.stat().st_size <= 0:
        raise RuntimeError(f"Figure was not written: {path}")

    with Image.open(path) as image:
        image_size = image.size

    if min(image_size) < 600:
        raise RuntimeError(
            f"Figure resolution is below the standard: "
            f"{path.name} -> {image_size}"
        )


required_history_columns = {"epoch", "loss", "val_loss", "auc", "val_auc"}
missing_history_columns = (
    required_history_columns - set(combined_history_df.columns)
)
if missing_history_columns:
    raise RuntimeError(
        f"Training history is missing columns: "
        f"{sorted(missing_history_columns)}"
    )

# 1) Training / validation loss
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(
    combined_history_df["epoch"],
    combined_history_df["loss"],
    label="Training Loss",
    linewidth=2,
)
ax.plot(
    combined_history_df["epoch"],
    combined_history_df["val_loss"],
    label="Validation Loss",
    linewidth=2,
    linestyle="--",
)
ax.set_title(
    "DenseNet121 Eyebrow ROI Training and Validation Loss",
    fontsize=14,
)
ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("Loss", fontsize=11)
ax.legend()
ax.grid(True, alpha=0.25)
save_figure(
    fig,
    FIGURE_DIR / "training_validation_loss.png",
)

# 2) AUC by epoch
fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(
    combined_history_df["epoch"],
    combined_history_df["auc"],
    label="Training ROC-AUC",
    linewidth=2,
)
ax.plot(
    combined_history_df["epoch"],
    combined_history_df["val_auc"],
    label="Validation ROC-AUC",
    linewidth=2,
    linestyle="--",
)
ax.set_title(
    "DenseNet121 Eyebrow ROI ROC-AUC by Epoch",
    fontsize=14,
)
ax.set_xlabel("Epoch", fontsize=11)
ax.set_ylabel("ROC-AUC", fontsize=11)
ax.set_ylim(0.0, 1.05)
ax.legend()
ax.grid(True, alpha=0.25)
save_figure(
    fig,
    FIGURE_DIR / "training_validation_auc.png",
)

# 3) Confusion matrix
cm = confusion_matrix(
    test_labels,
    test_predictions,
    labels=[0, 1],
)

fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
im = ax.imshow(cm)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            fontsize=14,
        )

ax.set_title(
    "Test Confusion Matrix — Eyebrow ROI",
    fontsize=14,
)
ax.set_xlabel("Predicted Class", fontsize=11)
ax.set_ylabel("True Class", fontsize=11)
ax.set_xticks([0, 1], labels=["Real", "Fake"])
ax.set_yticks([0, 1], labels=["Real", "Fake"])
fig.colorbar(im, ax=ax)
save_figure(
    fig,
    FIGURE_DIR / "test_confusion_matrix.png",
)

# 4) ROC curve
fpr, tpr, _ = roc_curve(
    test_labels,
    test_probabilities,
)

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(
    fpr,
    tpr,
    linewidth=2,
    label=f"ROC-AUC = {image_metrics['roc_auc']:.4f}",
)
ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1,
)
ax.set_title("Test ROC Curve — Eyebrow ROI", fontsize=14)
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.legend()
ax.grid(True, alpha=0.25)
save_figure(
    fig,
    FIGURE_DIR / "test_roc_curve.png",
)

# 5) Precision-Recall curve
precision_curve, recall_curve, _ = precision_recall_curve(
    test_labels,
    test_probabilities,
)

fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
ax.plot(
    recall_curve,
    precision_curve,
    linewidth=2,
    label=f"PR-AUC = {image_metrics['pr_auc']:.4f}",
)
ax.set_title(
    "Test Precision-Recall Curve — Eyebrow ROI",
    fontsize=14,
)
ax.set_xlabel("Recall", fontsize=11)
ax.set_ylabel("Precision", fontsize=11)
ax.legend()
ax.grid(True, alpha=0.25)
save_figure(
    fig,
    FIGURE_DIR / "test_precision_recall_curve.png",
)

figure_audit = []
for figure_path in sorted(FIGURE_DIR.glob("*.png")):
    with Image.open(figure_path) as image:
        width_px, height_px = image.size

    figure_audit.append(
        {
            "file": figure_path.name,
            "width_px": int(width_px),
            "height_px": int(height_px),
            "short_side_px": int(min(width_px, height_px)),
            "resolution_pass": bool(
                min(width_px, height_px) >= 600
            ),
        }
    )

figure_audit_df = pd.DataFrame(figure_audit)

if len(figure_audit_df) != 5:
    raise RuntimeError(
        f"Expected 5 standard figures, found {len(figure_audit_df)}."
    )
if not bool(figure_audit_df["resolution_pass"].all()):
    raise RuntimeError("At least one figure is below 600 px short side.")

figure_audit_df.to_csv(
    ARTIFACT_DIR / "figure_quality_audit.csv",
    index=False,
)

print(figure_audit_df.to_string(index=False))


                           file  width_px  height_px  short_side_px  resolution_pass
      test_confusion_matrix.png      1124       1185           1124             True
test_precision_recall_curve.png      1485        885            885             True
             test_roc_curve.png      1485        885            885             True
    training_validation_auc.png      1485        885            885             True
   training_validation_loss.png      1484        885            885             True


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [15]:
# ============================================================
# CELL 15 — FINAL AUDIT, SUMMARY AND OUTPUT MANIFEST
# ============================================================

if not STATE_PATH.exists():
    raise RuntimeError("training_state.json is missing.")

state = json.loads(
    STATE_PATH.read_text(encoding="utf-8")
)

best_metric_name = CONFIG["model_selection_metric"]
best_val_auc = float(
    state.get(best_metric_name, np.nan)
)
completed_epoch = int(
    state.get("completed_epoch", 0)
)

if not np.isfinite(best_val_auc):
    raise FloatingPointError(
        f"Best validation metric is not finite: {best_val_auc}"
    )


def json_safe_number(value):
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value)
    return value


run_summary = {
    "run_id": RUN_ID,
    "task": "Eyebrow ROI Deepfake Binary Classification",
    "model": "DenseNet121",
    "pretraining": "ImageNet",
    "input_root": str(EYEBROW_ROOT),
    "output_root": str(RUN_ROOT),
    "seed": SEED,
    "image_size": [IMAGE_HEIGHT, IMAGE_WIDTH],
    "resize_policy": CONFIG["resize_policy"],
    "split_policy": CONFIG["split_policy"],
    "train_images": int(len(train_df)),
    "validation_images": int(len(val_df)),
    "test_images": int(len(test_df)),
    "test_sources": int(len(source_predictions)),
    "best_validation_auc": float(best_val_auc),
    "completed_epoch": int(completed_epoch),
    "selected_validation_threshold": float(
        SELECTED_THRESHOLD
    ),
    "image_level_test_metrics": {
        key: json_safe_number(value)
        for key, value in image_metrics.items()
    },
    "source_level_test_metrics": {
        key: json_safe_number(value)
        for key, value in source_metrics.items()
    },
    "quality_gates": {
        key: bool(value)
        for key, value in quality_gates.items()
    },
    "leakage_pass": bool(leakage_pass),
    "test_used_for_model_selection": False,
}

with open(
    RUN_ROOT / "run_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        run_summary,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )

required_directories = [
    CHECKPOINT_DIR,
    LOG_DIR,
    METRICS_DIR,
    PREDICTION_DIR,
    FIGURE_DIR,
    ARTIFACT_DIR,
]

required_files = [
    RUN_ROOT / "config_resolved.yaml",
    RUN_ROOT / "environment.json",
    RUN_ROOT / "requirements_lock.txt",
    RUN_ROOT / "run_summary.json",
    BEST_MODEL_PATH,
    LAST_MODEL_PATH,
    STATE_PATH,
    MANIFEST_PATH,
    METRICS_DIR / "quality_gates.json",
    METRICS_DIR / "frozen_training_history.csv",
    METRICS_DIR / "finetune_training_history.csv",
    METRICS_DIR / "combined_training_history.csv",
    METRICS_DIR / "validation_threshold.json",
    METRICS_DIR / "final_image_level_metrics.csv",
    METRICS_DIR / "final_source_level_metrics.csv",
    METRICS_DIR / "classification_report.csv",
    PREDICTION_DIR / "test_image_predictions.csv",
    PREDICTION_DIR / "test_source_predictions.csv",
    ARTIFACT_DIR / "dataset_count_audit.csv",
    ARTIFACT_DIR / "source_group_split_assignment.csv",
    ARTIFACT_DIR / "leakage_check.json",
    ARTIFACT_DIR / "accounting_summary.json",
    ARTIFACT_DIR / "model_summary.txt",
    ARTIFACT_DIR / "figure_quality_audit.csv",
    FIGURE_DIR / "training_validation_loss.png",
    FIGURE_DIR / "training_validation_auc.png",
    FIGURE_DIR / "test_confusion_matrix.png",
    FIGURE_DIR / "test_roc_curve.png",
    FIGURE_DIR / "test_precision_recall_curve.png",
]

for directory in required_directories:
    if not directory.exists() or not directory.is_dir():
        raise RuntimeError(f"Missing output directory: {directory}")

for file_path in required_files:
    if not file_path.exists() or file_path.stat().st_size <= 0:
        raise RuntimeError(
            f"Missing/empty required output: {file_path}"
        )

final_audit = {
    "input_folder_untouched_by_design": True,
    "schema_gate": bool(quality_gates["schema_test"]),
    "source_split_gate": bool(quality_gates["split_test"]),
    "smoke_gate": bool(quality_gates["smoke_test"]),
    "numerical_gate": bool(quality_gates["numerical_test"]),
    "checkpoint_gate": bool(quality_gates["checkpoint_test"]),
    "test_only_final_evaluation": True,
    "all_figure_short_sides_ge_600px": bool(
        figure_audit_df["resolution_pass"].all()
    ),
    "english_figure_labels": True,
    "best_checkpoint_exists": bool(BEST_MODEL_PATH.exists()),
    "last_checkpoint_exists": bool(LAST_MODEL_PATH.exists()),
    "final_audit_pass": True,
    "output_manifest_note": (
        "output_manifest.csv is generated after final_audit.json "
        "and excludes only itself."
    ),
}

FINAL_AUDIT_PATH = ARTIFACT_DIR / "final_audit.json"

with open(
    FINAL_AUDIT_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        final_audit,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )


def build_output_manifest():
    records = []

    for file_path in sorted(
        path
        for path in RUN_ROOT.rglob("*")
        if path.is_file()
    ):
        if file_path.name == "output_manifest.csv":
            continue

        records.append(
            {
                "relative_path": str(
                    file_path.relative_to(RUN_ROOT)
                ),
                "size_bytes": int(file_path.stat().st_size),
                "sha256": sha256_file(file_path),
            }
        )

    frame = pd.DataFrame(
        records,
        columns=[
            "relative_path",
            "size_bytes",
            "sha256",
        ],
    )
    frame.to_csv(
        RUN_ROOT / "output_manifest.csv",
        index=False,
    )
    return frame


output_manifest_df = build_output_manifest()

if (
    RUN_ROOT / "output_manifest.csv"
).stat().st_size <= 0:
    raise RuntimeError("output_manifest.csv is empty.")

# final_audit.json must be represented in the output manifest.
if "artifacts/final_audit.json" not in set(
    output_manifest_df["relative_path"]
):
    raise RuntimeError(
        "final_audit.json is missing from output_manifest.csv."
    )

print("\n" + "=" * 88)
print("FINAL RUN SUMMARY")
print("=" * 88)
print("Run folder:", RUN_ROOT)
print("Best val AUC:", best_val_auc)
print("Test F1:", image_metrics["f1"])
print("Test ROC-AUC:", image_metrics["roc_auc"])
print("Test PR-AUC:", image_metrics["pr_auc"])
print("Leakage gate: PASS")
print("Final audit: PASS")
print("Output manifest: PASS")

print("\nCreated top-level outputs:")
for item in sorted(
    RUN_ROOT.iterdir(),
    key=lambda p: p.name.casefold(),
):
    print(" -", item.name)



FINAL RUN SUMMARY
Run folder: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/DenseNet121_Kas_Results_20260808_1335_eyebrow_densenet121_seed42
Best val AUC: 0.5752734541893005
Test F1: 0.24193548387096775
Test ROC-AUC: 0.554047992823503
Test PR-AUC: 0.5292930068489279
Leakage gate: PASS
Final audit: PASS
Output manifest: PASS

Created top-level outputs:
 - artifacts
 - checkpoints
 - config_resolved.yaml
 - environment.json
 - figures
 - logs
 - metrics
 - output_manifest.csv
 - predictions
 - requirements_lock.txt
 - run_summary.json


## Çalıştırma sırası

Notebook'u Colab'da **GPU runtime** ile aç ve hücreleri yukarıdan aşağı **Run all** çalıştır.

Beklenen kritik PASS mesajları:

```text
Metadata mapping        : PASS
Hash-aware group build : PASS
Source-group re-split  : PASS
Accounting equality     : PASS
Exact source isolation  : PASS
Source-group isolation  : PASS
Content leakage         : PASS
TF.DATA PIPELINE: PASS
ALL PRE-TRAINING QUALITY GATES: PASS
```

Bunlardan sonra DenseNet121 frozen eğitim ve fine-tuning otomatik devam eder. Girdi `Kaş` klasörüne yazma yapılmaz; yalnızca `Deney 1/Sonuçlar` içine yeni bir run klasörü oluşturulur.
